In [ ]:
# ================================================================
# RANDOM FINITE-AUTOMATA SAT SEARCH
# Google Colab — ONE CELL
#
# Upload DIMACS .cnf
# -> Generate random finite-state controllers
# -> Search for a satisfying assignment
# -> Periodic status updates without console spam
# -> Verify the witness
# -> Save + download SAT_solution.txt
# ================================================================

from google.colab import files
from IPython.display import clear_output

import random
import time
import os
from pathlib import Path


# ================================================================
# CONFIGURATION
# ================================================================

AUTOMATON_STATES = 32

# None = automatically chosen from CNF size
STEPS_PER_AUTOMATON = None

# None = continue until SAT is found
MAX_AUTOMATA = None

# Refresh only every N seconds -> avoids lag
UPDATE_SECONDS = 3.0

# Probability of using a random flip instead of automaton's
# preferred local strategy.
NOISE = 0.08

# None = random seed each execution
MASTER_SEED = None


# ================================================================
# UPLOAD CNF
# ================================================================

print("Upload a DIMACS .cnf file...")
uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No file uploaded.")

filename = next(iter(uploaded))
raw = uploaded[filename].decode("utf-8", errors="ignore")

clear_output(wait=True)

print("Loaded:", filename)
print("Parsing CNF...")


# ================================================================
# DIMACS PARSER
# ================================================================

def parse_dimacs(text):

    declared_vars = None
    declared_clauses = None

    clauses = []
    current = []

    for line in text.splitlines():

        line = line.strip()

        if not line:
            continue

        if line.startswith("c"):
            continue

        if line.startswith("%"):
            break

        if line.startswith("p"):

            parts = line.split()

            if len(parts) >= 4 and parts[1].lower() == "cnf":
                declared_vars = int(parts[2])
                declared_clauses = int(parts[3])

            continue

        for token in line.split():

            try:
                x = int(token)
            except ValueError:
                continue

            if x == 0:

                # Normalize repeated literals.
                unique = set(current)

                # Tautological clause:
                # (x OR NOT x) is always true, so remove it.
                tautology = any(-lit in unique for lit in unique)

                if not tautology:
                    clauses.append(list(unique))

                current = []

            else:
                current.append(x)

    if current:
        unique = set(current)

        if not any(-lit in unique for lit in unique):
            clauses.append(list(unique))

    inferred_vars = 0

    for clause in clauses:
        for lit in clause:
            inferred_vars = max(inferred_vars, abs(lit))

    nvars = max(declared_vars or 0, inferred_vars)

    return nvars, clauses, declared_clauses


N_VARS, CLAUSES, DECLARED_CLAUSES = parse_dimacs(raw)

if N_VARS == 0:
    raise RuntimeError("Could not detect variables in CNF.")

if any(len(c) == 0 for c in CLAUSES):
    raise RuntimeError(
        "The CNF contains an empty clause, which directly implies UNSAT."
    )

N_CLAUSES = len(CLAUSES)
N_LITERALS = sum(len(c) for c in CLAUSES)

print(
    f"Variables: {N_VARS:,}\n"
    f"Clauses:   {N_CLAUSES:,}\n"
    f"Literals:  {N_LITERALS:,}"
)


# ================================================================
# INDEXED SET
#
# Allows:
#   add/remove O(1)
#   indexed access O(1)
#
# Useful because the set of unsatisfied clauses changes constantly.
# ================================================================

class IndexedSet:

    def __init__(self):
        self.items = []
        self.pos = {}

    def add(self, x):

        if x in self.pos:
            return

        self.pos[x] = len(self.items)
        self.items.append(x)

    def remove(self, x):

        idx = self.pos.pop(x, None)

        if idx is None:
            return

        last = self.items.pop()

        if idx < len(self.items):
            self.items[idx] = last
            self.pos[last] = idx

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        return self.items[i]


# ================================================================
# BUILD OCCURRENCE LIST
# ================================================================

OCCURRENCES = [[] for _ in range(N_VARS + 1)]

for ci, clause in enumerate(CLAUSES):

    for lit in clause:

        v = abs(lit)

        OCCURRENCES[v].append(
            (
                ci,
                lit > 0
            )
        )


# ================================================================
# SAT STATE
# ================================================================

class SATState:

    def __init__(self, rng):

        self.assignment = [False] * (N_VARS + 1)

        for v in range(1, N_VARS + 1):
            self.assignment[v] = bool(rng.getrandbits(1))

        self.sat_count = [0] * N_CLAUSES

        self.unsat = IndexedSet()

        for ci, clause in enumerate(CLAUSES):

            count = 0

            for lit in clause:

                v = abs(lit)
                value = self.assignment[v]

                if lit < 0:
                    value = not value

                if value:
                    count += 1

            self.sat_count[ci] = count

            if count == 0:
                self.unsat.add(ci)

    def break_count(self, variable):

        count = 0
        value = self.assignment[variable]

        for ci, positive in OCCURRENCES[variable]:

            literal_true = (
                value if positive
                else not value
            )

            if literal_true and self.sat_count[ci] == 1:
                count += 1

        return count

    def flip(self, variable):

        old_value = self.assignment[variable]

        for ci, positive in OCCURRENCES[variable]:

            before = (
                old_value if positive
                else not old_value
            )

            old_count = self.sat_count[ci]

            if before:
                new_count = old_count - 1
            else:
                new_count = old_count + 1

            self.sat_count[ci] = new_count

            if old_count == 0 and new_count > 0:
                self.unsat.remove(ci)

            elif old_count > 0 and new_count == 0:
                self.unsat.add(ci)

        self.assignment[variable] = not old_value


# ================================================================
# RANDOM FINITE AUTOMATON
#
# Input alphabet:
#
# 0 = search improved
# 1 = unchanged
# 2 = search worsened
#
# Each state contains a random search action.
# ================================================================

def random_automaton(rng, q):

    # delta[state][observation] -> next state
    delta = [
        [
            rng.randrange(q)
            for _ in range(3)
        ]
        for _ in range(q)
    ]

    # Action selected by each state
    #
    # 0 = random literal
    # 1 = minimum break-count literal
    # 2 = fixed literal position
    # 3 = choose among two random literals
    action = [
        rng.randrange(4)
        for _ in range(q)
    ]

    # Used to choose which unsatisfied clause is inspected
    clause_salt = [
        rng.getrandbits(64)
        for _ in range(q)
    ]

    # Used by action 2
    literal_slot = [
        rng.randrange(65536)
        for _ in range(q)
    ]

    return {
        "delta": delta,
        "action": action,
        "clause_salt": clause_salt,
        "literal_slot": literal_slot
    }


# ================================================================
# SELECT VARIABLE USING AUTOMATON STATE
# ================================================================

def choose_variable(
    automaton,
    automaton_state,
    sat_state,
    step,
    rng
):

    U = sat_state.unsat

    # Select an unsatisfied clause using the automaton state.
    index = (
        automaton["clause_salt"][automaton_state]
        + step * 0x9E3779B97F4A7C15
    ) % len(U)

    clause_index = U[index]
    clause = CLAUSES[clause_index]

    variables = [abs(lit) for lit in clause]

    mode = automaton["action"][automaton_state]

    # Small random noise helps escape local minima.
    if rng.random() < NOISE:
        return rng.choice(variables)

    # ------------------------------------------------------------
    # MODE 0
    # Random variable from violated clause
    # ------------------------------------------------------------

    if mode == 0:
        return rng.choice(variables)

    # ------------------------------------------------------------
    # MODE 1
    # Minimum break-count
    # ------------------------------------------------------------

    if mode == 1:

        scores = [
            (
                sat_state.break_count(v),
                v
            )
            for v in variables
        ]

        best_score = min(x[0] for x in scores)

        best = [
            v
            for score, v in scores
            if score == best_score
        ]

        return rng.choice(best)

    # ------------------------------------------------------------
    # MODE 2
    # Fixed automaton output slot
    # ------------------------------------------------------------

    if mode == 2:

        slot = (
            automaton["literal_slot"][automaton_state]
            % len(variables)
        )

        return variables[slot]

    # ------------------------------------------------------------
    # MODE 3
    # Tournament between two random literals
    # ------------------------------------------------------------

    a = rng.choice(variables)
    b = rng.choice(variables)

    break_a = sat_state.break_count(a)
    break_b = sat_state.break_count(b)

    if break_a < break_b:
        return a

    if break_b < break_a:
        return b

    return rng.choice([a, b])


# ================================================================
# VERIFY FINAL ASSIGNMENT
# ================================================================

def verify_assignment(assignment):

    for clause in CLAUSES:

        clause_ok = False

        for lit in clause:

            value = assignment[abs(lit)]

            if lit < 0:
                value = not value

            if value:
                clause_ok = True
                break

        if not clause_ok:
            return False

    return True


# ================================================================
# SAVE DIMACS-STYLE SOLUTION
# ================================================================

def save_solution(
    assignment,
    automaton,
    automaton_number,
    steps,
    seed,
    elapsed
):

    path = "/content/SAT_solution.txt"

    literals = []

    for v in range(1, N_VARS + 1):

        if assignment[v]:
            literals.append(v)
        else:
            literals.append(-v)

    with open(path, "w") as f:

        f.write(
            "c Random Finite-Automata SAT Search\n"
        )

        f.write(
            f"c source {filename}\n"
        )

        f.write(
            f"c variables {N_VARS}\n"
        )

        f.write(
            f"c clauses {N_CLAUSES}\n"
        )

        f.write(
            f"c automaton_number {automaton_number}\n"
        )

        f.write(
            f"c automaton_states {AUTOMATON_STATES}\n"
        )

        f.write(
            f"c automaton_steps {steps}\n"
        )

        f.write(
            f"c random_seed {seed}\n"
        )

        f.write(
            f"c elapsed_seconds {elapsed:.6f}\n"
        )

        f.write(
            "c witness independently verified against every clause\n"
        )

        f.write("\n")

        f.write("s SATISFIABLE\n")

        # Standard DIMACS-style witness.
        CHUNK = 30

        for i in range(0, len(literals), CHUNK):

            chunk = literals[i:i + CHUNK]

            suffix = " 0" if i + CHUNK >= len(literals) else ""

            f.write(
                "v "
                + " ".join(map(str, chunk))
                + suffix
                + "\n"
            )

        f.write("\n")
        f.write("c FINITE AUTOMATON\n")

        delta = automaton["delta"]
        actions = automaton["action"]

        action_names = {
            0: "random-literal",
            1: "min-break",
            2: "fixed-slot",
            3: "two-literal-tournament"
        }

        for q in range(AUTOMATON_STATES):

            f.write(
                f"c q{q}: "
                f"improve->q{delta[q][0]} "
                f"same->q{delta[q][1]} "
                f"worse->q{delta[q][2]} "
                f"action={action_names[actions[q]]}\n"
            )

    return path


# ================================================================
# AUTO PARAMETERS
# ================================================================

if STEPS_PER_AUTOMATON is None:

    # Enough room for exploration without letting one automaton
    # monopolize the run forever.
    STEPS = max(
        2_000,
        min(
            250_000,
            40 * N_VARS
        )
    )

else:
    STEPS = int(STEPS_PER_AUTOMATON)


# ================================================================
# MASTER RNG
# ================================================================

if MASTER_SEED is None:
    MASTER_SEED = int.from_bytes(
        os.urandom(8),
        "little"
    )

master_rng = random.Random(MASTER_SEED)


# ================================================================
# SEARCH
# ================================================================

start_time = time.time()
last_update = 0.0

automata_tested = 0
total_steps = 0

best_unsat = N_CLAUSES
best_automaton = None


def status(
    current_unsat=None,
    force=False
):

    global last_update

    now = time.time()

    if (
        not force
        and now - last_update < UPDATE_SECONDS
    ):
        return

    last_update = now

    elapsed = now - start_time

    rate = (
        automata_tested / elapsed
        if elapsed > 0
        else 0
    )

    clear_output(wait=True)

    print(
        "╔════════════════════════════════════════════════╗"
    )

    print(
        "║       RANDOM FINITE-AUTOMATA SAT SEARCH        ║"
    )

    print(
        "╚════════════════════════════════════════════════╝"
    )

    print()

    print(f"File:                 {filename}")
    print(f"Variables:            {N_VARS:,}")
    print(f"Clauses:              {N_CLAUSES:,}")
    print(f"Literals:             {N_LITERALS:,}")
    print()

    print(f"Automaton states:     {AUTOMATON_STATES}")
    print(f"Steps / automaton:    {STEPS:,}")
    print()

    print(f"Automata tested:      {automata_tested:,}")
    print(f"Transitions tested:   {total_steps:,}")
    print(f"Best unsat clauses:   {best_unsat:,}")

    if current_unsat is not None:
        print(
            f"Current unsat:        {current_unsat:,}"
        )

    print()

    print(
        f"Elapsed:              {elapsed:.1f} s"
    )

    print(
        f"Automata / second:    {rate:.2f}"
    )

    print()

    print(
        "Searching..."
    )


status(force=True)


# ================================================================
# MAIN LOOP
# ================================================================

solution = None

try:

    while True:

        if (
            MAX_AUTOMATA is not None
            and automata_tested >= MAX_AUTOMATA
        ):
            break

        automata_tested += 1

        automaton_seed = master_rng.getrandbits(64)

        rng = random.Random(
            automaton_seed
        )

        automaton = random_automaton(
            rng,
            AUTOMATON_STATES
        )

        sat_state = SATState(rng)

        controller_state = rng.randrange(
            AUTOMATON_STATES
        )

        local_best = len(
            sat_state.unsat
        )

        # Immediate random hit.
        if local_best == 0:

            solution = (
                sat_state,
                automaton,
                automaton_seed,
                0
            )

            break

        for step in range(1, STEPS + 1):

            old_unsat = len(
                sat_state.unsat
            )

            variable = choose_variable(
                automaton,
                controller_state,
                sat_state,
                step,
                rng
            )

            sat_state.flip(variable)

            total_steps += 1

            new_unsat = len(
                sat_state.unsat
            )

            # Observation fed back into finite automaton.
            if new_unsat < old_unsat:
                observation = 0

            elif new_unsat == old_unsat:
                observation = 1

            else:
                observation = 2

            controller_state = (
                automaton["delta"]
                [controller_state]
                [observation]
            )

            if new_unsat < local_best:
                local_best = new_unsat

            if new_unsat < best_unsat:

                best_unsat = new_unsat
                best_automaton = automata_tested

            # ----------------------------------------------------
            # SAT FOUND
            # ----------------------------------------------------

            if new_unsat == 0:

                solution = (
                    sat_state,
                    automaton,
                    automaton_seed,
                    step
                )

                break

            status(
                current_unsat=new_unsat
            )

        if solution is not None:
            break


except KeyboardInterrupt:

    clear_output(wait=True)

    print(
        "Search manually stopped."
    )

    print(
        f"Automata tested: {automata_tested:,}"
    )

    print(
        f"Best unsatisfied-clause count: {best_unsat:,}"
    )


# ================================================================
# RESULT
# ================================================================

if solution is not None:

    sat_state, automaton, seed, steps = solution

    assignment = sat_state.assignment

    print("Verifying witness...")

    verified = verify_assignment(
        assignment
    )

    if not verified:

        raise RuntimeError(
            "Internal verification failed."
        )

    elapsed = (
        time.time()
        - start_time
    )

    output_path = save_solution(
        assignment,
        automaton,
        automata_tested,
        steps,
        seed,
        elapsed
    )

    clear_output(wait=True)

    print(
        "╔════════════════════════════════════════════════╗"
    )

    print(
        "║                 SAT FOUND                     ║"
    )

    print(
        "╚════════════════════════════════════════════════╝"
    )

    print()

    print(
        f"File:               {filename}"
    )

    print(
        f"Variables:          {N_VARS:,}"
    )

    print(
        f"Clauses:            {N_CLAUSES:,}"
    )

    print(
        f"Automaton #:        {automata_tested:,}"
    )

    print(
        f"States:             {AUTOMATON_STATES}"
    )

    print(
        f"Steps in winner:    {steps:,}"
    )

    print(
        f"Total transitions:  {total_steps:,}"
    )

    print(
        f"Elapsed:            {elapsed:.3f} s"
    )

    print()

    print(
        "Witness verification: PASS ✓"
    )

    print()

    preview = []

    for v in range(
        1,
        min(N_VARS, 20) + 1
    ):

        preview.append(
            f"x{v}="
            + (
                "1"
                if assignment[v]
                else "0"
            )
        )

    print(
        "Assignment preview:"
    )

    print(
        " ".join(preview)
    )

    if N_VARS > 20:
        print("...")

    print()

    print(
        "Saved:"
    )

    print(
        output_path
    )

    # Automatic download.
    files.download(output_path)


elif MAX_AUTOMATA is not None:

    clear_output(wait=True)

    print(
        "No satisfying assignment was found within the configured limit."
    )

    print(
        f"Automata tested: {automata_tested:,}"
    )

    print(
        f"Best unsatisfied-clause count: {best_unsat:,}"
    )

    print()

    print(
        "This does NOT prove UNSAT."
    )